# plotlywaterfall examples

Waterfall charts with [plotly](https://plotly.com/python/) for financial reporting: bridges from one level to another, stacked categories, groups side by side, totals and subtotals, value labels.
Every example is one call to `waterfall(...)`, which returns a normal `plotly.graph_objects.Figure`.

**Part 1** shows typical finance charts, **part 2** every feature on its own.

Source: [github.com/docdru/plotlywaterfall](https://github.com/docdru/plotlywaterfall) · install with `pip install plotlywaterfall`

In [ ]:
import pandas as pd
import plotly.io as pio

from plotlywaterfall import Waterfall, waterfall

pio.templates.default = "plotly_white"

## Part 1: Finance examples

### 1.1 EBIT bridge from last year to this year

The classic bridge: start at last year's level, show the drivers, end at this year's level. `start` marks the x-value whose value is a level (grey, unsigned label), not a change; `total=True` adds the end level.

In [ ]:
ebit = pd.DataFrame({
    "item": ["EBIT 2025", "Volume", "Price / mix", "FX", "Personnel", "Material", "Other opex"],
    "MEUR": [100.0, 12.4, 5.1, -3.8, -6.2, -2.9, -1.5],
})
waterfall(
    ebit, x="item", y="MEUR",
    start="EBIT 2025", total=True, total_column_name="EBIT 2026",
    number_suffix=" M€", title="EBIT bridge 2025 → 2026", labels={"MEUR": "EBIT [MEUR]"},
)

The levels dwarf the drivers. Reports often zoom the value axis instead of starting at zero; that is plain plotly:

In [ ]:
fig = waterfall(
    ebit, x="item", y="MEUR", start="EBIT 2025", total=True, total_column_name="EBIT 2026",
    number_suffix=" M€", labels={"MEUR": "EBIT [MEUR]"},
)
fig.update_yaxes(range=[90, 120])

### 1.2 Budget vs actual by segment

When the data says what each row is, use a `measure` column (the names follow plotly's `go.Waterfall`): `"absolute"` for given levels such as budget and actual, `"relative"` for changes, `"total"` for calculated levels.

Budget plus the explained variances rarely equals the actual exactly. By default (`reconcile="other"`) the difference becomes an explicit **Other** bar, so the walk adds up to every given level; here it differs per segment, so the bar is split into sub-steps:

In [ ]:
bva = pd.DataFrame({
    "item":    ["Budget", "Budget", "Volume", "Volume", "Price", "Price", "Costs", "Costs", "Actual", "Actual"],
    "segment": ["Industrial", "Consumer"] * 5,
    "MEUR":    [60, 40, 5, -3, 2, 0.5, -1.5, -2, 64, 38],
    "measure": ["absolute"] * 2 + ["relative"] * 6 + ["absolute"] * 2,
})
waterfall(bva, x="item", y="MEUR", category="segment", measure="measure", title="Budget vs actual")

`reconcile="jump"` draws the given levels as they are (the connector then visibly jumps), `reconcile="error"` raises an error instead:

In [ ]:
try:
    waterfall(bva, x="item", y="MEUR", category="segment", measure="measure", reconcile="error")
except ValueError as exc:
    print("ValueError:", exc)

### 1.3 Monthly cash-flow walk

Dates on the x-axis are shown as months, `number_unit="auto"` writes all labels in one unit and `negative_style="parentheses"` gives accounting-style negatives:

In [ ]:
months = pd.date_range("2026-01-31", periods=6, freq="ME")
cash = pd.DataFrame({
    "month": ["Opening", *months],
    "EUR": [1_200_000, 150_000, -320_000, 80_000, -40_000, 210_000, -60_000],
})
waterfall(
    cash, x="month", y="EUR", start="Opening", total=True, total_column_name="Closing",
    number_unit="auto", negative_style="parentheses", number_prefix="€",
    title="Cash H1 2026",
)

### 1.4 One bridge per region, side by side

With `group`, each region gets its own bar next to the others, starting from its own level:

In [ ]:
regions = pd.DataFrame({
    "item":   ["EBIT 2025", "Volume", "Price", "Costs"] * 2,
    "region": ["Europe"] * 4 + ["Americas"] * 4,
    "MEUR":   [60, 4, 2, -5, 40, 7, -1, -2],
})
waterfall(
    regions, x="item", y="MEUR", group="region",
    start="EBIT 2025", total=True, total_column_name="EBIT 2026",
    color="region",
)

### 1.5 From a ledger to a bridge

Report data usually starts as a long table of postings. Sum it per driver with pandas, collect the small drivers into one bar, and pass the result. `category_orders` fixes the order of the bars:

In [ ]:
ledger = pd.DataFrame({
    "driver": ["Volume", "Volume", "Price", "FX", "Freight", "Energy", "Personnel", "Personnel", "Insurance", "IT"],
    "amount": [7.5, 4.9, 5.1, -3.8, -0.4, -0.9, -4.0, -2.2, -0.3, 0.2],
})
by_driver = ledger.groupby("driver", as_index=False)["amount"].sum()
small = by_driver["amount"].abs() < 1
by_driver.loc[small, "driver"] = "Other items"
by_driver = by_driver.groupby("driver", as_index=False)["amount"].sum()

bridge = pd.concat([pd.DataFrame({"driver": ["EBIT 2025"], "amount": [100.0]}), by_driver])
waterfall(
    bridge, x="driver", y="amount", start="EBIT 2025", total=True, total_column_name="EBIT 2026",
    category_orders={"driver": ["EBIT 2025", "Volume", "Price", "FX", "Personnel", "Other items"]},
)

## Part 2: Features

### 2.1 A basic waterfall

Pass a DataFrame and the names of the x and y columns. Rows with the same `X` are summed. Without categories or groups, bars are coloured by direction (increase, decrease, total) and labelled with their change:

In [ ]:
df_cash = pd.DataFrame({
    "X": ["Product Sales", "Product Sales", "Services Sales", "Product R&D", "Product R&D", "Services COGS"],
    "Y": [3, 4, 1, -1, -2, -0.5],
    "category": ["Product A", "Product B", "Plugins", "Product A", "Product B", "Plugins"],
})
df_cash

In [ ]:
waterfall(df_cash, x="X", y="Y", labels={"Y": "Cashflow [MUSD]"})

### 2.2 Totals and subtotals

`total=True` adds a total bar after the last x-value; `subtotals` adds intermediate totals after the given x-values:

In [ ]:
waterfall(
    df_cash, x="X", y="Y", total=True, subtotals={"Services Sales": "Gross revenue"},
    labels={"Y": "Cashflow [MUSD]"},
)

With the class `Waterfall`, subtotals can also be added afterwards with `add_subtotal`. The total can be negative:

In [ ]:
df_loss = df_cash.assign(Y=[3, 4, 1, -4, -5, -2])
wf = Waterfall(df_loss, x="X", y="Y", total=True, labels={"Y": "Cashflow [MUSD]"})
wf.add_subtotal("Services Sales", "Gross revenue")
wf.get_fig()

### 2.3 Stacked categories

With `category`, each bar is stacked from its categories, and totals keep the breakdown:

In [ ]:
waterfall(df_cash, x="X", y="Y", category="category", total=True, labels={"Y": "Cashflow [MUSD]"})

### 2.4 Mixed signs within one bar

Categories of one bar can have different signs, e.g. a subsidy (+1) and a restructuring charge (−4) at the same `X`, or a total combining positive and negative categories. Such a bar is split into one narrow sub-step per category, walking from the level where the bar starts to where it ends. A light *net ghost* bar behind the sub-steps shows the net change:

In [ ]:
df_mixed = pd.DataFrame({
    "X": ["Revenue", "Costs", "Other", "Other"],
    "Y": [9, -7, 1, -4],
    "category": ["Operating", "Operating", "Operating", "One-off"],
})
waterfall(df_mixed, x="X", y="Y", category="category", total=True)

The order of the sub-steps follows the data; `step_order="positive_first"` or `"largest_first"` changes it. `net_ghost=False` hides the net bar (a dict of plotly marker properties styles it):

In [ ]:
waterfall(df_mixed, x="X", y="Y", category="category", total=True, step_order="largest_first", net_ghost=False)

`mixed_signs="error"` restores the strict behaviour of earlier versions:

In [ ]:
try:
    Waterfall(df_mixed, x="X", y="Y", category="category", mixed_signs="error")
except NotImplementedError as exc:
    print("NotImplementedError:", exc)

### 2.5 Groups

`group` draws one bar per group side by side, e.g. to compare plan and actual or several years. This DataFrame is used in the following sections:

In [ ]:
df = pd.DataFrame({
    "X": ["A", "B", "C"] * 4 + ["D", "D"],
    "Y": [4, 1, 8, 7, 3, 2] + [i - 1 for i in [4, 1, 8, 7, 3, 2]] + [8, 5],
    "category": ["one"] * 3 + ["two"] * 3 + ["one"] * 3 + ["two"] * 3 + ["two", "three"],
    "group": ["Group1"] * 6 + ["Group2"] * 6 + ["Group2", "Group3"],
})
df

In [ ]:
waterfall(df, x="X", y="Y", category="category", group="group", total=True, subtotals={"C": "Subtotal"})

`total_type="sumtotal"` adds one plain total bar per group instead of a stacked total:

In [ ]:
waterfall(df, x="X", y="Y", category="category", group="group", total=True, total_type="sumtotal")

### 2.6 Colours

The `color` argument says what drives the colours: the category column (default if there is one), the group column, or `"direction"` (increase / decrease / total; default for charts without category and group). Colours are assigned per value, so a category has the same colour in every group.

`color_discrete_map` sets colours per value; values not in the map get colours from `color_discrete_sequence` (default: the colorway of the plotly template). `pattern_shape` adds fill patterns as a second visual cue, e.g. colour by category and pattern by group:

In [ ]:
waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color_discrete_map={"one": "#2a78d6", "two": "#eb6834", "three": "#1baf7a"},
    pattern_shape="group",
)

Colour by group instead:

In [ ]:
waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color="group",
    color_discrete_map={"Group1": "steelblue", "Group2": "darkorange", "Group3": "seagreen"},
)

For full control, keys can also be `(group, category)` tuples. They take precedence over plain values:

In [ ]:
colors = {
    "one": "red", "two": "blue", "three": "green",
    ("Group2", "one"): "salmon", ("Group2", "two"): "lightskyblue",
}
waterfall(df, x="X", y="Y", category="category", group="group", total=True, color_discrete_map=colors)

Direction colours also work with categories and groups. Combine them with a pattern so that the categories stay distinguishable; `color_discrete_map` keys `"increase"`, `"decrease"` and `"total"` override the default green / red / grey:

In [ ]:
waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    color="direction", pattern_shape="category",
)

Keys of `color_discrete_map` that match no value raise a warning, so typos do not go unnoticed.

Coming from version 0.9? The `colors` argument was replaced by `color` / `color_discrete_map`:

| 0.9 `colors=` | 1.0 |
|---|---|
| `{"one": "red", "two": "blue"}` (per category) | `color_discrete_map={"one": "red", "two": "blue"}` |
| `{"Group1": "red", "Group2": "blue"}` (per group) | `color="group", color_discrete_map={"Group1": "red", "Group2": "blue"}` |
| `{"Group1": {"one": "red"}}` (per group and category) | `color_discrete_map={("Group1", "one"): "red"}` |

### 2.7 Labels, number format and hover

By default every bar is labelled with its change (signed, e.g. `+9`) or, for totals, its value. Numbers get thousands separators and up to two decimals. `number_prefix` / `number_suffix` add units (the sign stays in front: `-$5M`), `number_format` takes a Python format spec such as `",.1f"` or a function. `title` and `labels` work like in plotly express. Hover over a bar to see the step, its category and the running total.

In [ ]:
waterfall(
    df_cash, x="X", y="Y", total=True,
    number_prefix="$", number_suffix="M",
    title="Cash bridge", labels={"Y": "Cashflow [MUSD]", "X": "Line item"},
)

A function as `number_format`, here to show thousands as "k":

In [ ]:
df_k = df_cash.assign(Y=df_cash.Y * 1000)
waterfall(df_k, x="X", y="Y", total=True, number_format=lambda v: f"{v / 1000:.1f}k", number_prefix="€")

`number_unit` (`"k"`, `"M"`, `"bn"` or `"auto"`) scales all numbers to one unit, `negative_style="parentheses"` writes negatives as in accounting:

In [ ]:
waterfall(df_k, x="X", y="Y", total=True, number_unit="k", negative_style="parentheses", number_prefix="€")

`value_labels="segment"` labels every stacked segment instead (segments too small for their label stay unlabelled), `"both"` shows both kinds and `None` hides all labels:

In [ ]:
waterfall(df, x="X", y="Y", category="category", group="group", total=True, value_labels="segment")

### 2.8 Order and dates

Bars follow the order of appearance in the data, or the order of a pandas `Categorical`. `category_orders` (as in plotly express) sets the order of x, group or category values explicitly; listed values come first:

In [ ]:
waterfall(df_cash, x="X", y="Y", total=True, category_orders={"X": ["Services Sales", "Product Sales"]})

Dates are labelled as months when they fall in different months; `x_format` takes a strftime pattern or a function:

In [ ]:
quarters = pd.DataFrame({"quarter": pd.date_range("2026-03-31", periods=4, freq="QE"), "Y": [5, -2, 3, 1]})
waterfall(quarters, x="quarter", y="Y", total=True, x_format=lambda d: f"Q{d.quarter} {d.year}")

### 2.9 Layout and styling

`barwidth`, `gap` (between the bars of different groups) and `connector_line` (plotly line properties) change the geometry. The result is a regular plotly figure, so everything else — size, legend, fonts, templates — is set with plotly's own methods:

In [ ]:
fig = waterfall(
    df, x="X", y="Y", category="category", group="group", total=True,
    barwidth=0.25, gap=0.05,
    connector_line={"color": "black", "width": 1, "dash": "dot"},
)
fig.update_layout(template="simple_white", height=450, legend={"orientation": "h", "y": -0.15})

### 2.10 Export for reports

The figure is a regular plotly figure: show it in Jupyter, Dash or Quarto, or export it for slides and PDFs (static export needs the `kaleido` package):

```python
fig = waterfall(ebit, x="item", y="MEUR", start="EBIT 2025", total=True, total_column_name="EBIT 2026")
fig.write_image("ebit_bridge.svg", width=900, height=450)   # also .png, .pdf
fig.write_html("ebit_bridge.html", include_plotlyjs="cdn")  # interactive, self-contained page
```